# 🏥 Health-InsurTech — Prédire les frais de santé de manière éthique
## Étape 2 : Modélisation Interprétable & Analyse des Biais

**Objectif** : entraîner un modèle interprétable (Régression Linéaire + Arbre de Décision) pour estimer les frais médicaux annuels d'un client, tout en analysant et atténuant les biais potentiels.

---
| Section | Contenu |
|---|---|
| 0 | Imports & configuration |
| 1 | Chargement & nettoyage éthique des données |
| 2 | Split Train / Test |
| 3 | Modèle 1 — Régression Linéaire |
| 4 | Modèle 2 — Arbre de Décision |
| 5 | Audit des biais |
| 6 | Atténuation des biais (Ridge + Calibration) |
| 7 | Récapitulatif & Recommandations RGPD |

## 0. Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor, export_text, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Palette graphique
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]
plt.rcParams["figure.dpi"] = 120
print("✅ Imports OK")

✅ Imports OK


## 1. Chargement & Nettoyage Éthique des Données

> **Principe de minimisation (RGPD Art. 5)** : seules les variables strictement nécessaires à la prédiction sont conservées. Toutes les données à caractère personnel identifiable (PII) sont exclues du modèle.

In [ ]:
df = pd.read_csv("raw_data/insurance_data.csv")
print(f"Dataset chargé : {df.shape[0]} lignes × {df.shape[1]} colonnes")
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: 'insurance_data.csv'

In [ ]:
# ── Colonnes retenues pour la modélisation (pas de PII) ──────────────────
FEATURES = ["age", "bmi", "children", "smoker", "region", "sex"]
TARGET   = "charges"

# ── Colonnes PII supprimées ───────────────────────────────────────────────
PII_COLS = [
    "id_client", "nom", "prenom", "date_naissance", "email",
    "telephone", "numero_secu_sociale", "ville", "code_postal",
    "adresse_ip", "date_inscription", "region_fr",
    "mutuelle_complementaire", "consentement_rgpd"
]

df_model = df[FEATURES + [TARGET]].copy()
df_model.dropna(subset=FEATURES + [TARGET], inplace=True)

print(f"✅ PII supprimées ({len(PII_COLS)} colonnes)")
print(f"✅ Lignes utilisables : {df_model.shape[0]}")
df_model.describe().round(2)

In [ ]:
# ── Encodage one-hot des variables catégorielles ──────────────────────────
df_enc = pd.get_dummies(df_model, columns=["smoker", "region", "sex"], drop_first=False)

# Suppression des colonnes redondantes (éviter la multicolinéarité)
# Références : smoker_no, sex_male, region_southwest
for col in ["smoker_no", "sex_male", "region_southwest"]:
    if col in df_enc.columns:
        df_enc.drop(columns=[col], inplace=True)

feature_cols = [c for c in df_enc.columns if c != TARGET]
X = df_enc[feature_cols]
y = df_enc[TARGET]

print(f"Features finales ({len(feature_cols)}) :")
for f in feature_cols:
    print(f"  • {f}")

## 2. Split Train / Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standardisation (nécessaire pour la régression linéaire)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train : {len(X_train)} échantillons")
print(f"Test  : {len(X_test)} échantillons")

# Distribution des charges
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y_train, bins=40, color=PALETTE[0], edgecolor="white", alpha=0.85)
axes[0].set_title("Distribution des charges (Train)", fontweight="bold")
axes[0].set_xlabel("Frais annuels (€)")
axes[0].set_ylabel("Fréquence")
axes[1].boxplot([y_train, y_test], labels=["Train", "Test"], patch_artist=True,
                boxprops=dict(facecolor=PALETTE[2], alpha=0.6))
axes[1].set_title("Charges : Train vs Test", fontweight="bold")
axes[1].set_ylabel("Frais annuels (€)")
plt.tight_layout()
plt.show()

## 3. Modèle 1 — Régression Linéaire

La régression linéaire produit des **coefficients directement interprétables** : chaque coefficient indique l'impact d'une variable sur les frais (en €, à features standardisées).

In [ ]:
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)

mae_lr  = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr   = r2_score(y_test, y_pred_lr)

print(f"📊 Régression Linéaire")
print(f"   MAE  : {mae_lr:>10,.2f} €")
print(f"   RMSE : {rmse_lr:>10,.2f} €")
print(f"   R²   : {r2_lr:>10.4f}")

In [ ]:
coef_df = pd.DataFrame({
    "Feature"    : feature_cols,
    "Coefficient": lr.coef_
}).sort_values("Coefficient", ascending=False)

print("Coefficients (features standardisées) :")
display(coef_df.style.bar(subset=["Coefficient"], align="zero",
                           color=["#4C72B0", "#C44E52"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Coefficients
colors = [PALETTE[3] if v > 0 else PALETTE[0] for v in coef_df["Coefficient"]]
axes[0].barh(coef_df["Feature"], coef_df["Coefficient"], color=colors, edgecolor="white")
axes[0].axvline(0, color="black", lw=0.8)
axes[0].set_title("Coefficients — Régression Linéaire\n(features standardisées)", fontweight="bold")
axes[0].set_xlabel("Coefficient (€)")
pos_p = mpatches.Patch(color=PALETTE[3], label="↑ Augmente les frais")
neg_p = mpatches.Patch(color=PALETTE[0], label="↓ Réduit les frais")
axes[0].legend(handles=[pos_p, neg_p], fontsize=9)

# Réel vs Prédit
axes[1].scatter(y_test, y_pred_lr, alpha=0.4, color=PALETTE[0], s=14)
lims = [y_test.min(), y_test.max()]
axes[1].plot(lims, lims, "r--", lw=1.5, label="Prédiction parfaite")
axes[1].set_title("Réel vs Prédit — Régression Linéaire", fontweight="bold")
axes[1].set_xlabel("Charges réelles (€)")
axes[1].set_ylabel("Charges prédites (€)")
axes[1].legend()
axes[1].text(0.05, 0.92, f"R² = {r2_lr:.3f}", transform=axes[1].transAxes,
             bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow"), fontsize=10)

plt.tight_layout()
plt.show()

## 4. Modèle 2 — Arbre de Décision

L'arbre de décision produit des **règles lisibles** directement auditables par un expert métier. On limite la profondeur à 4 pour garder le modèle interprétable.

In [ ]:
dt = DecisionTreeRegressor(max_depth=4, min_samples_leaf=20, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

mae_dt  = mean_absolute_error(y_test, y_pred_dt)
rmse_dt = np.sqrt(mean_squared_error(y_test, y_pred_dt))
r2_dt   = r2_score(y_test, y_pred_dt)

print(f"🌳 Arbre de Décision (max_depth=4)")
print(f"   MAE  : {mae_dt:>10,.2f} €")
print(f"   RMSE : {rmse_dt:>10,.2f} €")
print(f"   R²   : {r2_dt:>10.4f}")

In [ ]:
print("Règles de l'arbre de décision :")
print(export_text(dt, feature_names=feature_cols, max_depth=4))

In [ ]:
fi_df = pd.DataFrame({
    "Feature"   : feature_cols,
    "Importance": dt.feature_importances_
}).sort_values("Importance", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Importances
axes[0].barh(fi_df["Feature"], fi_df["Importance"], color=PALETTE[2], edgecolor="white")
axes[0].set_title("Importance des features\nArbre de Décision", fontweight="bold")
axes[0].set_xlabel("Importance (réduction d'impureté Gini)")

# Visualisation de l'arbre
plot_tree(dt, feature_names=feature_cols, filled=True, rounded=True,
          max_depth=3, fontsize=7, ax=axes[1])
axes[1].set_title("Arbre de décision (depth=3 affiché)", fontweight="bold")

plt.tight_layout()
plt.show()

## 5. Audit des Biais — Analyse d'Équité

> **Objectif** : détecter si le modèle sur-pénalise certaines catégories (fumeurs, régions, sexe).
> On mesure l'**erreur moyenne par groupe** : si elle dépasse **+15%** des charges réelles, c'est un signal de sur-pénalisation.

In [ ]:
# DataFrame de test enrichi
df_test = X_test.copy()
df_test["charges_reel"]  = y_test.values
df_test["pred_lr"]       = y_pred_lr
df_test["pred_dt"]       = y_pred_dt
df_test["erreur_lr"]     = df_test["pred_lr"] - df_test["charges_reel"]
df_test["erreur_abs_lr"] = df_test["erreur_lr"].abs()

# Labels lisibles
df_test["Fumeur"] = df_test["smoker_yes"].astype(int).map({1: "Fumeur", 0: "Non-fumeur"})
df_test["Sexe"]   = df_test["sex_female"].astype(int).map({1: "Femme", 0: "Homme"})

region_map = {"region_northeast": "northeast", "region_northwest": "northwest",
               "region_southeast": "southeast"}
def get_region(row):
    for col, name in region_map.items():
        if col in row.index and row[col] == 1:
            return name
    return "southwest"
df_test["Region"] = df_test.apply(get_region, axis=1)

print("✅ Jeu de test annoté")

In [ ]:
print("► Biais par statut fumeur")
bias_smoker = df_test.groupby("Fumeur")[["charges_reel", "pred_lr",
                                          "erreur_lr", "erreur_abs_lr"]].mean().round(2)
display(bias_smoker)

In [ ]:
print("► Biais par région")
bias_region = df_test.groupby("Region")[["charges_reel", "pred_lr",
                                          "erreur_lr", "erreur_abs_lr"]].mean().round(2)
display(bias_region)

In [ ]:
print("► Biais par sexe")
bias_sex = df_test.groupby("Sexe")[["charges_reel", "pred_lr",
                                      "erreur_lr", "erreur_abs_lr"]].mean().round(2)
display(bias_sex)

In [ ]:
print("► Détection automatique de sur-pénalisation (seuil +15%)\n")
for group_name, bias_df in [("Fumeur", bias_smoker), ("Région", bias_region), ("Sexe", bias_sex)]:
    for idx, row in bias_df.iterrows():
        biais_pct = (row["erreur_lr"] / row["charges_reel"]) * 100
        if biais_pct > 15:
            flag = "⚠️  SUR-PÉNALISATION"
        elif abs(biais_pct) < 5:
            flag = "✅ OK"
        else:
            flag = "ℹ️  À surveiller"
        print(f"  {flag} [{group_name}] {idx:15s} — biais moyen : {biais_pct:+.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Audit des biais — Erreur de prédiction par groupe",
             fontsize=13, fontweight="bold")

# Fumeur
groups_sm = ["Non-fumeur", "Fumeur"]
reel_sm   = [df_test[df_test["Fumeur"] == g]["charges_reel"].mean() for g in groups_sm]
pred_sm   = [df_test[df_test["Fumeur"] == g]["pred_lr"].mean()      for g in groups_sm]
x = np.arange(2)
axes[0].bar(x - 0.2, reel_sm, 0.35, label="Réel",    color=PALETTE[0], alpha=0.85)
axes[0].bar(x + 0.2, pred_sm, 0.35, label="Prédit",  color=PALETTE[1], alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(groups_sm)
axes[0].set_title("Par statut fumeur", fontweight="bold")
axes[0].set_ylabel("Frais moyens (€)"); axes[0].legend(fontsize=8)

# Région — boxplot des erreurs
regions = sorted(df_test["Region"].unique())
data_r  = [df_test[df_test["Region"] == r]["erreur_lr"].values for r in regions]
bp = axes[1].boxplot(data_r, labels=regions, patch_artist=True, widths=0.55)
for patch, color in zip(bp["boxes"], PALETTE):
    patch.set_facecolor(color); patch.set_alpha(0.65)
axes[1].axhline(0, color="red", lw=1.2, linestyle="--", label="Biais nul")
axes[1].set_title("Distribution erreurs par région", fontweight="bold")
axes[1].set_ylabel("Erreur (€)"); axes[1].legend(fontsize=8)
axes[1].tick_params(axis="x", rotation=15)

# Sexe
groups_sx = ["Femme", "Homme"]
reel_sx   = [df_test[df_test["Sexe"] == g]["charges_reel"].mean() for g in groups_sx]
pred_sx   = [df_test[df_test["Sexe"] == g]["pred_lr"].mean()      for g in groups_sx]
x2 = np.arange(2)
axes[2].bar(x2 - 0.2, reel_sx, 0.35, label="Réel",   color=PALETTE[0], alpha=0.85)
axes[2].bar(x2 + 0.2, pred_sx, 0.35, label="Prédit", color=PALETTE[4], alpha=0.85)
axes[2].set_xticks(x2); axes[2].set_xticklabels(groups_sx)
axes[2].set_title("Par sexe", fontweight="bold")
axes[2].set_ylabel("Frais moyens (€)"); axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 6. Atténuation des Biais

Trois leviers complémentaires :

| Levier | Description |
|---|---|
| **Ridge (L2)** | Pénalise les gros coefficients → réduit l'influence disproportionnée de `smoker_yes` |
| **Calibration par groupe** | Corrige le biais résiduel de chaque sous-groupe via un offset calculé sur le train |
| **Exclusion des PII** | Aucune donnée personnelle n'entre dans le modèle (RGPD) |

In [ ]:
# ── Ridge Regression ─────────────────────────────────────────────────────
ridge = Ridge(alpha=10.0)
ridge.fit(X_train_sc, y_train)
y_pred_ridge = ridge.predict(X_test_sc)

mae_r  = mean_absolute_error(y_test, y_pred_ridge)
rmse_r = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
r2_r   = r2_score(y_test, y_pred_ridge)

print(f"📊 Ridge (alpha=10)")
print(f"   MAE  : {mae_r:>10,.2f} €")
print(f"   RMSE : {rmse_r:>10,.2f} €")
print(f"   R²   : {r2_r:>10.4f}")

# Comparaison des coefficients LR vs Ridge
coef_compare = pd.DataFrame({
    "Feature"         : feature_cols,
    "Coef LR"         : lr.coef_,
    "Coef Ridge (L2)" : ridge.coef_
}).sort_values("Coef LR", ascending=False)

display(coef_compare.style.background_gradient(cmap="RdYlGn_r",
        subset=["Coef LR", "Coef Ridge (L2)"]))

In [ ]:
# ── Calibration par groupe (post-processing) ─────────────────────────────
X_train_df = pd.DataFrame(X_train_sc, columns=feature_cols, index=X_train.index)
X_train_df["pred_ridge"]   = ridge.predict(X_train_sc)
X_train_df["charges_reel"] = y_train.values
X_train_df["erreur"]       = X_train_df["pred_ridge"] - X_train_df["charges_reel"]
X_train_df["Fumeur"]       = X_train["smoker_yes"].astype(int).values

offsets = X_train_df.groupby("Fumeur")["erreur"].mean()
print(f"Offsets de calibration calculés sur le train :")
print(f"  Non-fumeur : {offsets.get(0, 0):+.2f} €")
print(f"  Fumeur     : {offsets.get(1, 0):+.2f} €")

# Application sur le test
df_test["pred_ridge"]     = y_pred_ridge
df_test["smoker_int"]     = df_test["smoker_yes"].astype(int)
df_test["pred_ridge_cal"] = df_test.apply(
    lambda row: row["pred_ridge"] - offsets.get(int(row["smoker_int"]), 0), axis=1
)

mae_cal = mean_absolute_error(df_test["charges_reel"], df_test["pred_ridge_cal"])
r2_cal  = r2_score(df_test["charges_reel"], df_test["pred_ridge_cal"])
print(f"\n📊 Ridge + Calibration")
print(f"   MAE : {mae_cal:,.0f} €   R² : {r2_cal:.4f}")

In [ ]:
# Biais résiduel après calibration
bias_after = df_test.groupby("smoker_int").apply(
    lambda g: ((g["pred_ridge_cal"] - g["charges_reel"]) / g["charges_reel"]).mean() * 100
).round(2)

print("Biais résiduel après calibration :")
print(f"  Non-fumeur : {bias_after.get(0, float('nan')):+.2f}%")
print(f"  Fumeur     : {bias_after.get(1, float('nan')):+.2f}%")

# Visualisation avant / après
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
groups = ["Non-fumeur", "Fumeur"]
key_map = {"Non-fumeur": 0, "Fumeur": 1}

reel   = [df_test[df_test["smoker_int"] == key_map[g]]["charges_reel"].mean()     for g in groups]
p_lr   = [df_test[df_test["smoker_int"] == key_map[g]]["pred_lr"].mean()          for g in groups]
p_cal  = [df_test[df_test["smoker_int"] == key_map[g]]["pred_ridge_cal"].mean()   for g in groups]

x = np.arange(2)
for ax, pred, title in zip(axes, [p_lr, p_cal],
                            ["Avant atténuation (Rég. Linéaire)",
                             "Après atténuation (Ridge + Cal.)"]):
    ax.bar(x - 0.2, reel, 0.35, label="Réel",    color=PALETTE[0], alpha=0.85)
    ax.bar(x + 0.2, pred, 0.35, label="Prédit",  color=PALETTE[1], alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(groups)
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel("Frais moyens (€)"); ax.legend(fontsize=8)

plt.suptitle("Effet de la calibration sur le biais fumeur",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Récapitulatif & Recommandations RGPD

In [ ]:
results = pd.DataFrame({
    "Modèle": [
        "Régression Linéaire",
        "Arbre de Décision (depth=4)",
        "Ridge (alpha=10)",
        "Ridge + Calibration"
    ],
    "MAE (€)": [mae_lr, mae_dt, mae_r, mae_cal],
    "RMSE (€)": [rmse_lr, rmse_dt, rmse_r, None],
    "R²": [r2_lr, r2_dt, r2_r, r2_cal],
    "Interprétable": ["✅", "✅", "✅", "✅"],
    "Anti-biais": ["❌", "❌", "⚠️", "✅"]
})

results["MAE (€)"]  = results["MAE (€)"].apply(lambda x: f"{x:,.0f} €" if x else "—")
results["RMSE (€)"] = results["RMSE (€)"].apply(lambda x: f"{x:,.0f} €" if x else "—")
results["R²"]       = results["R²"].apply(lambda x: f"{x:.4f}")

print("Récapitulatif des modèles :")
display(results.set_index("Modèle"))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
model_names = ["Lin. Reg.", "Dec. Tree", "Ridge", "Ridge+Cal"]
maes = [mae_lr, mae_dt, mae_r, mae_cal]
bar_colors = [PALETTE[3] if m == min(maes) else PALETTE[0] for m in maes]
bars = ax.bar(model_names, maes, color=bar_colors, edgecolor="white", width=0.5)
ax.set_title("MAE comparée entre modèles (€ — plus bas = meilleur)", fontweight="bold")
ax.set_ylabel("MAE (€)")
for bar, val in zip(bars, maes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f"{val:,.0f}€", ha="center", va="bottom", fontsize=9)
best_patch = mpatches.Patch(color=PALETTE[3], label="Meilleure MAE")
ax.legend(handles=[best_patch])
plt.tight_layout()
plt.show()

## ✅ Recommandations Éthiques & RGPD

### Mesures appliquées dans ce notebook

| Mesure | Détail |
|---|---|
| 🔒 **PII exclues** | Nom, email, numéro de sécu, IP, adresse — aucune PII dans le modèle |
| 🔍 **Transparence** | Coefficients LR et règles de l'arbre entièrement auditables |
| ⚖️ **Audit des biais** | Erreur mesurée par fumeur, région et sexe |
| 🛠️ **Atténuation** | Ridge (L2) + calibration par groupe corrige le biais résiduel |

### Recommandations supplémentaires

- **Intervalles de confiance** : afficher une fourchette (± X €) à l'utilisateur plutôt qu'un chiffre sec
- **Non-discrimination** : ne jamais refuser un contrat sur la seule base du modèle
- **Réentraînement** : mettre à jour le modèle au moins une fois par an avec de nouvelles données
- **Audit Fairness** : utiliser [Fairlearn](https://fairlearn.org/) ou [AIF360](https://aif360.mybluemix.net/) pour un audit formel annuel
- **Consentement explicite** : les données de santé (statut fumeur = donnée sensible RGPD Art. 9) nécessitent un consentement explicite